# Stateful Jupyter Interactive Session Verification Guide

This notebook serves as an interactive playground to verify the modular **Jupyter gRPC Service** implemented inside the `agent-sandbox-agent` daemon.

You will run cell executions locally in this notebook, which will dynamically send code payloads over gRPC to compile and execute inside a secure sandbox container running on your GKE cluster.

---

## Step 1: Install Client Dependencies
First, install the necessary gRPC client packages inside your local notebook runtime:

In [ ]:
# Install gRPC and gRPC tools
!pip install grpcio grpcio-tools

## Step 2: Compile Protobuf Python Client Stubs

Compile the Protobuf definition file (`api/proto/v1/agent_sandbox.proto`) into Python stubs. 

*(Make sure you are running this notebook inside your repository path, or upload the `agent_sandbox.proto` file to your Colab files sidebar).*

In [ ]:
# Compile the Protobuf file to generate Python client stubs
!python3 -m grpc_tools.protoc \
    -I./api/proto/v1 \
    --python_out=./examples \
    --grpc_python_out=./examples \
    api/proto/v1/agent_sandbox.proto

print("Python stubs compiled successfully inside examples folder!")

## Step 3: Establish Connection & Spawn Stateful Session

Now, connect to the port-forwarded GKE sandbox container (`localhost:50051`) and create a persistent, stateful Python kernel session.

In [ ]:
import sys
import os
import grpc

# Ensure generated stubs are in python search path
sys.path.append(os.path.abspath("./examples"))

import agent_sandbox_pb2 as pb
import agent_sandbox_pb2_grpc as pb_grpc

# Connect to the GKE sandbox container
# Replace with target sandbox Pod IP if not using port-forwarding
channel = grpc.insecure_channel("localhost:50051")
jupyter_stub = pb_grpc.JupyterServiceStub(channel)

# Spawn a stateful Python 3 kernel session
session = jupyter_stub.CreateSession(pb.CreateJupyterSessionRequest(kernel_name="python3"))
session_id = session.session_id
print(f"Successfully spawned stateful GKE Sandbox Jupyter Session!")
print(f"Session ID: {session_id}")

## Step 4: E2E Test - Stateful Variable Persistence

Now, let's execute two sequential code blocks. 

In the first block, we declare variables and import modules inside the sandbox kernel memory. 

In the second block, we read those variables. This proves the stateful persistence of the GKE runtime!

In [ ]:
# Execution Cell 1: Declare states inside the remote sandbox kernel memory
code_1 = """
import math
radius = 10
# Calculate circle area
circle_area = math.pi * (radius ** 2)
"""

res1 = jupyter_stub.ExecuteCode(pb.ExecuteJupyterCodeRequest(
    session_id=session_id,
    code=code_1
))
print(f"Cell 1 Execution Status: {res1.status}")
print("Stdout:", res1.stdout)
print("Stderr:", res1.stderr)

In [ ]:
# Execution Cell 2: Read and print variables defined in Cell 1
# (Proves persistence)
code_2 = """
print(f"Calculated Circle Area: {circle_area:.4f}")
"""

res2 = jupyter_stub.ExecuteCode(pb.ExecuteJupyterCodeRequest(
    session_id=session_id,
    code=code_2
))
print(f"Cell 2 Execution Status: {res2.status}")
print(f"Remote Sandbox Stdout:\n{res2.stdout.strip()}")

## Step 5: Administrative Sandbox Clean

Wipe all local directories and forcefully terminate any user-spawned background processes, resetting the sandbox environment cleanly.

In [ ]:
# Call admin Clean
admin_stub = pb_grpc.AdminServiceStub(channel)
admin_stub.Clean(pb.CleanRequest())
print("Sandbox wiped clean successfully.")